# Crawl chính thức 20.000 ID — chạy theo ca (Colab CPU-only)

Owner: Hiếu (Member 3) · soạn hộ: Thiên, 27/10/2026. Soạn với sự hỗ trợ của AI assistant (Claude, Anthropic), ghi nguồn theo quy định môn học.

**Quy tắc (plan, mục 2.3 và 3.4):**

- Chỉ **một** ca chạy tại một thời điểm, vì arXiv chỉ cho một kết nối trên mọi máy của nhóm. Notebook ghi dấu `SHIFT.json` trên Drive chung và **từ chối chạy** nếu ca của người khác còn hoạt động trong 45 phút gần nhất.
- Corpus và sổ sách (`data/`, `work/`) nằm trên Drive chung. Ca mới chép về ổ máy Colab, chạy, và đồng bộ ngược lên Drive sau mỗi `CHUNK` ID.
- Bị ngắt phiên: mở lại notebook, **Run all**. Crawler tự bỏ qua version đã xong.
- Kết thúc ca: chạy ô cuối cùng để đồng bộ và nhả ca, rồi báo số liệu vào nhóm chat.

Trước khi chạy: Colab Secrets có `S2_API_KEY` (key Semantic Scholar) và `ARXIV_CONTACT_EMAIL`; thư mục `DRIVE_DIR` đã được share cho cả nhóm.

In [ ]:
# Tham số của ca: chỉ sửa ô này
SHIFT_OWNER = "thien"  # người trực ca: thien | tien | hieu | dang
RANGES = ["thien", "tien", "hieu", "dang"]  # thứ tự crawl, một instance cho cả 4 dải
RANGE_SIZE = 5000
CHUNK = 500  # đồng bộ Drive sau mỗi CHUNK ID
DRIVE_DIR = "/content/drive/MyDrive/arxiv-nhom1"  # thư mục chung của nhóm
REPO = "https://github.com/hcmus-arxiv-nhom1/CodeRepo.git"
BRANCH = "main"
REFRESH_METADATA = False  # True cho ca ĐẦU TIÊN của lần crawl chính thức (plan, mục 1.3)
FORCE = False  # chỉ True khi chắc chắn ca trước đã dừng hẳn

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess

if os.path.isdir("/content/CodeRepo/.git"):
    subprocess.run(["git", "-C", "/content/CodeRepo", "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/CodeRepo"], check=True)
%cd /content/CodeRepo
!pip install -q -r src/requirements.txt
!git log --oneline -1

In [ ]:
from google.colab import userdata

for name in ("S2_API_KEY", "ARXIV_CONTACT_EMAIL"):
    try:
        os.environ[name] = userdata.get(name)
        print(f"{name}: đã đặt")
    except Exception:  # secret chưa tạo hoặc chưa cấp quyền cho notebook
        print(f"{name}: CHƯA đặt trong Colab Secrets")

In [ ]:
# Dấu bàn giao ca trên Drive: không cho hai ca chạy cùng lúc
import json
import time
from datetime import UTC, datetime
from pathlib import Path

DRIVE = Path(DRIVE_DIR)
LOCAL = Path("/content/run")
MARKER = DRIVE / "SHIFT.json"
for d in (DRIVE / "data" / "corpus", DRIVE / "work", LOCAL / "data" / "corpus", LOCAL / "work"):
    d.mkdir(parents=True, exist_ok=True)


def heartbeat(released=False):
    now = datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ")
    MARKER.write_text(
        json.dumps({"owner": SHIFT_OWNER, "at": time.time(), "at_iso": now, "released": released})
    )


if MARKER.exists() and not FORCE:
    m = json.loads(MARKER.read_text())
    age_min = (time.time() - m["at"]) / 60
    if m["owner"] != SHIFT_OWNER and not m.get("released") and age_min < 45:
        raise RuntimeError(
            f"Ca của {m['owner']} còn hoạt động {age_min:.0f} phút trước. "
            "Chờ họ chạy ô 'Kết thúc ca', hoặc hỏi trong nhóm chat."
        )
heartbeat()
print("Nhận ca:", SHIFT_OWNER)

In [ ]:
# Chép corpus và sổ sách từ Drive về ổ máy Colab (nhanh hơn ghi trực tiếp lên Drive)
!rsync -a {DRIVE_DIR}/data/ /content/run/data/
!rsync -a --exclude tmp/ {DRIVE_DIR}/work/ /content/run/work/
!du -sh /content/run/data /content/run/work

In [ ]:
import sys


def sync_to_drive():
    subprocess.run(["rsync", "-a", f"{LOCAL}/data/", f"{DRIVE}/data/"], check=True)
    subprocess.run(
        ["rsync", "-a", "--exclude", "tmp/", f"{LOCAL}/work/", f"{DRIVE}/work/"], check=True
    )
    heartbeat()


for name in RANGES:
    for offset in range(0, RANGE_SIZE, CHUNK):
        cmd = [sys.executable, "-m", "src.run_pipeline", "--range", name]
        cmd += ["--offset", str(offset), "--limit", str(CHUNK)]
        cmd += ["--data-root", f"{LOCAL}/data/corpus", "--work-dir", f"{LOCAL}/work"]
        cmd += ["--no-progress"]
        if REFRESH_METADATA:
            cmd.append("--refresh-metadata")
        print(f"=== {name} {offset}..{offset + CHUNK - 1}", flush=True)
        code = subprocess.run(cmd).returncode
        sync_to_drive()
        summary = json.loads((LOCAL / "work" / "run_summary.json").read_text())
        print({k: summary[k] for k in ("papers_complete", "ids_requested", "failed_rows")})
        if code == 2:
            raise RuntimeError("work/ đang bị một lần chạy khác giữ (run.lock)")
        if code != 0:
            raise RuntimeError(f"run_pipeline dừng với mã {code}: xem {LOCAL}/work/run.log")
print("Đã chạy hết 4 dải.")

## Kết thúc ca

Chạy ô dưới khi hết ca, kể cả khi crawl chưa xong: đồng bộ lần cuối và nhả ca cho người kế tiếp.

In [ ]:
sync_to_drive()
heartbeat(released=True)
!tail -n 3 /content/run/work/run.log
print("Đã nhả ca. Báo trong nhóm chat: số version đã xong, failed_rows, RAM/disk nếu có.")